# 01 — Extract the Anthropic S-1 into CSVs

**Question:** what does Anthropic's S-1 actually report, in a form we can compute on?

**Short answer:** every financial table we need, from the 5-period statements (FY2023–FY2025, nine months to
Sep 30 2025 / 2026) and the 7-quarter P&L to the convertible notes, preferred stack and $314B of compute
purchase commitments, now sits as tidy CSVs in `data/`. Reconciliation checks at the end pass.

| § | What | Output |
|---|---|---|
| 1 | Download + cache the filing | `source_docs/anthropic_424b4.htm` |
| 2 | Parse every HTML table | in-memory `T` |
| 3 | Statements → tidy CSVs | `data/*.csv` |
| 4 | Narrative KPIs (hand-entered, with source lines) | `data/kpis.csv` |
| 5 | Reconciliation checks | asserts |

**Source:** final prospectus (Rule 424(b)(4)), filed 2026-11-19, accession 0001628280-26-074944, priced at $83.00.
Its financial tables are identical to S-1/A No. 1 (filed 2026-11-02), which earlier runs used; only the
table positions moved (by 10, from the offering-terms tables now filled in at the front) and one label lost a typo.
Offering terms (shares sold, proceeds) live in `data/valuation_inputs.csv` as primary outside data.

**Conventions** (same as the SpaceX project): long format, `value` in millions USD unless `unit` says otherwise;
parentheses → negative; `—` → 0; `*` (below disclosure threshold) → empty.
Periods: `FY2025`, `9M-2026` (nine months to Sep 30), `Q3-2026`, balance-sheet dates as `2026-09-30`.

## 1. Download and cache the filing
SEC requires a contact email in the User-Agent. We only hit EDGAR if the cached copy is missing.

In [1]:
import os
import re
from pathlib import Path
import pandas as pd
import requests
from bs4 import BeautifulSoup

URL = 'https://www.sec.gov/Archives/edgar/data/2130357/000162828026074944/anthropicpbc-424b4.htm'
SRC = Path('source_docs/anthropic_424b4.htm')
DATA = Path('data')
# SEC wants a contact User-Agent; keep it out of git in .env (SEC_USER_AGENT=...)
if Path('.env').exists():
    for line in Path('.env').read_text().splitlines():
        if '=' in line and not line.lstrip().startswith('#'):
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))
UA = {'User-Agent': os.environ.get('SEC_USER_AGENT', 'set SEC_USER_AGENT in .env')}

if not SRC.exists():
    r = requests.get(URL, headers=UA, timeout=60)
    r.raise_for_status()
    SRC.write_bytes(r.content)
soup = BeautifulSoup(SRC.read_text(encoding='utf-8'), 'html.parser')
# plain-text copy for grepping the narrative
Path('source_docs/s1_full.txt').write_text(re.sub(r'\n\s*\n+', '\n', soup.get_text('\n')))
print(f'{SRC.stat().st_size/1e6:.1f} MB, {len(soup.find_all("table"))} tables')

14.4 MB, 162 tables


## 2. Parse every table
Each `<table>` becomes a list of rows of cell text. The filing splits `$`, `)` and `%` into their own cells, so we drop those.
Tables are addressed by their index `i` in the document; §5's checks would fail if an amendment shifted them.

In [2]:
def rows_of(t):
    out = []
    for tr in t.find_all('tr'):
        cells = [re.sub(r'\s+', ' ', td.get_text(' ')).strip() for td in tr.find_all(['td', 'th'])]
        cells = [c for c in cells if c and c not in ('$', ')', '%')]
        if cells:
            out.append(cells)
    return out

T = {i: rows_of(t) for i, t in enumerate(soup.find_all('table'))}

NUM = re.compile(r'^\(?-?[\d,]*\.?\d+\)?\s*%?$')
def is_num(s):
    s = s.replace('$', '').strip()
    return s in ('—', '*') or bool(NUM.match(s))

def num(s):
    s = s.replace('$', '').replace(',', '').replace('%', '').strip()
    if s == '—': return 0.0
    if s == '*': return None
    neg = s.startswith('(')
    v = float(s.strip('()'))
    return -v if neg else v

def clean(label):
    label = label.split(', $0.00001 par value')[0]  # drop the legal share-count text on equity lines
    return re.sub(r'(\s*\(\d\))+$', '', label).replace(' (unaudited)', '').strip()  # drop footnote markers

def tidy(i, periods, unit='millions_usd', skip=0):
    """Rows whose last len(periods) cells are numbers -> long records. Text-only rows become `section`."""
    n, out, section = len(periods), [], ''
    for r in T[i][skip:]:
        tail = r[-n:]
        if len(r) > n and all(is_num(c) for c in tail) and not r[0].startswith('(in'):
            label = clean(' '.join(r[:-n]))
            u = unit
            if '%' in ''.join(tail): u = 'pct'
            elif 'per share' in label.lower(): u = 'usd_per_share'
            elif 'shares outstanding' in label.lower(): u = 'thousands_shares'
            out += [dict(section=section, line_item=label, period=p, unit=u, value=num(v)) for p, v in zip(periods, tail)]
        elif len(r) == 1 and not is_num(r[0]) and not r[0].startswith('('):
            section = r[0].rstrip(':')
    return pd.DataFrame(out)

ptype = lambda p: 'year' if p.startswith('FY') else 'ytd_9m' if p.startswith('9M') else 'quarter' if p.startswith('Q') else 'instant'
def save(df, name):
    df = df.copy()
    if 'period' in df: df.insert(df.columns.get_loc('period') + 1, 'period_type', df['period'].map(ptype))
    df.to_csv(DATA / f'{name}.csv', index=False)
    print(f'{name}.csv: {len(df)} rows')
    return df

P5 = ['FY2023', 'FY2024', 'FY2025', '9M-2025', '9M-2026']
P3_BS = ['2024-12-31', '2025-12-31', '2026-09-30']
Q7 = ['Q1-2025', 'Q2-2025', 'Q3-2025', 'Q4-2025', 'Q1-2026', 'Q2-2026', 'Q3-2026']

## 3. Statements and schedules → tidy CSVs
### 3a. Core statements
Income statement (F-pages table 104), balance sheet (102–103), cash flows (109–110), comprehensive loss (105),
and the 7-quarter P&L from MD&A (58).

In [3]:
inc = save(tidy(104, P5), 'income_statement')
# the filing has no 'noncurrent' sub-headers, so its section labels would mislead: drop them
bs = save(pd.concat([tidy(102, P3_BS), tidy(103, P3_BS)]).drop(columns='section'), 'balance_sheet')
cf = save(pd.concat([tidy(109, P5), tidy(110, P5)]), 'cash_flow')
save(tidy(105, P5), 'comprehensive_income')
q = save(tidy(58, Q7), 'quarterly_income_statement')

income_statement.csv: 85 rows
balance_sheet.csv: 84 rows
cash_flow.csv: 155 rows
comprehensive_income.csv: 25 rows
quarterly_income_statement.csv: 105 rows


### 3b. Non-GAAP reconciliations
Adjusted Operating Income and Adjusted Net Income, annual (56, 57) and quarterly (60, 61). The big add-back is the
fair-value change on convertible notes: a non-cash mark that tracks Anthropic's own valuation, not operations.

In [4]:
ng = pd.concat([
    tidy(56, P5).assign(measure='adjusted_operating_income'),
    tidy(60, Q7).assign(measure='adjusted_operating_income'),
    tidy(57, P5).assign(measure='adjusted_net_income'),
    tidy(61, Q7).assign(measure='adjusted_net_income'),
])
ng = save(ng[['measure', 'line_item', 'period', 'unit', 'value']], 'non_gaap')

non_gaap.csv: 192 rows


### 3c. Revenue, expenses and customers
Revenue by geography (114) and by pricing model (115); opex by GAAP function (142, which reclassifies the
natural-expense lines into cost of revenue / R&D / S&M / G&A); customer concentration (111, 122; customers are anonymised letters).

In [5]:
rev = save(pd.concat([tidy(114, P5).assign(dimension='geography'),
                      tidy(115, P5).assign(dimension='pricing_model')])
           [['dimension', 'line_item', 'period', 'unit', 'value']], 'revenue_breakdown')
save(tidy(142, P5), 'opex_by_function')
conc = pd.concat([tidy(112, P5, unit='pct').assign(metric='share_of_revenue'),
                  tidy(111, P3_BS, unit='pct').assign(metric='share_of_receivables')])
save(conc.rename(columns={'line_item': 'customer'})[['metric', 'customer', 'period', 'unit', 'value']], 'customer_concentration');

revenue_breakdown.csv: 30 rows
opex_by_function.csv: 30 rows
customer_concentration.csv: 26 rows


### 3d. Balance-sheet and P&L detail schedules
One file each, with a `schedule` column: PP&E (131), prepaid (137), other noncurrent assets (138), payables/accruals (139)
for the balance sheet; lease cost (132), other income/expense (140), supplemental cash flow (141), tax provision (151),
credit-loss allowance (113) for the flows.

In [6]:
bsd = pd.concat([tidy(i, P3_BS).assign(schedule=s) for i, s in
                 [(131, 'ppe'), (137, 'prepaid_current'), (138, 'other_assets_noncurrent'), (139, 'payables_accruals')]])
save(bsd[['schedule', 'line_item', 'period', 'unit', 'value']], 'balance_sheet_detail')
fd = pd.concat([tidy(132, P5).assign(schedule='lease_cost'),
                tidy(140, P5).assign(schedule='other_income_expense'),
                tidy(141, P5).assign(schedule='supplemental_cash_flow'),
                tidy(151, P5[:3]).assign(schedule='tax_provision'),
                tidy(113, ['FY2024', 'FY2025', '9M-2026']).assign(schedule='credit_loss_allowance')])
save(fd[['schedule', 'section', 'line_item', 'period', 'unit', 'value']], 'flow_detail');

balance_sheet_detail.csv: 57 rows
flow_detail.csv: 154 rows


### 3e. Commitments and financing
- **Contractual obligations** (68): $320B total, almost all of it compute purchase commitments.
- **Convertible notes**: the Amazon rollover notes outstanding (62) and the undrawn 2026 facilities from Google, Amazon and Broadcom (63).
  Note the balance sheet carries the notes at **fair value** ($130.5B), far above the $6.2B principal.
- **Fair-value roll-forward** (125) for the Level 3 instruments (notes, warrants, partner financing, donation matching).
- **Deferred revenue roll-forward** (116).

In [7]:
yrs = ['total', '2026', '2027', '2028', '2029', '2030', 'thereafter']
co = tidy(68, yrs).rename(columns={'period': 'due'}).drop(columns='section')
co.insert(0, 'as_of', '2026-09-30')
co.to_csv(DATA / 'contractual_obligations.csv', index=False); print('contractual_obligations.csv:', len(co), 'rows')

def wide(i, header_row=0):
    """Small reference tables with mixed text/number columns: keep wide, clean headers and numbers."""
    rows = [[clean(c) or None for c in r] for r in T[i][header_row:]]
    cols, body = rows[0], [r for r in rows[1:] if r[0] and not r[0].startswith(('(', 'Total'))]
    n = max(len(r) for r in body)
    if len(cols) == n - 1: cols = ['item'] + cols  # header row has no label column
    df = pd.DataFrame([r + [None] * (n - len(r)) for r in body], columns=cols)
    for c in df.columns[1:]:
        if df[c].dropna().map(is_num).all(): df[c] = df[c].map(lambda x: num(x) if isinstance(x, str) else None)
    return df

notes_out = wide(62); notes_out.to_csv(DATA / 'convertible_notes_outstanding.csv', index=False)
facilities = wide(63); facilities.to_csv(DATA / 'note_facilities_2026.csv', index=False)
display(notes_out, facilities)

fv = wide(125).melt(id_vars=wide(125).columns[0], var_name='instrument', value_name='value')
fv = fv.rename(columns={fv.columns[0]: 'event'}).assign(unit='millions_usd')
fv['event'] = fv['event'].map(clean)
fv.to_csv(DATA / 'fair_value_rollforward.csv', index=False); print('fair_value_rollforward.csv:', len(fv), 'rows')

dr = pd.DataFrame([dict(event=clean(r[0]), value=num(r[1])) for r in T[116] if len(r) == 2], ).assign(unit='millions_usd')
dr.to_csv(DATA / 'deferred_revenue_rollforward.csv', index=False); print('deferred_revenue_rollforward.csv:', len(dr), 'rows')

contractual_obligations.csv: 28 rows


,Counterparty,Agreement,Principal Amount Outstanding (in millions),Conversion Price Per Share,Class N Shares Issuable on Conversion
0,Amazon,Amazon 2023 NPA,2186.0,2.0928,1.044717e+09
1,Amazon,Amazon 2024 NPA,4017.0,5.0478,7.957280e+08


,Counterparty,Agreement,Aggregate Amount Available for Issuance (in millions),Principal Amount Outstanding (in millions),Draw Period Ends
0,Google,Google 2026 NPA,20000.0,0.0,December 2030
1,Amazon,Amazon 2026 NPA,15000.0,0.0,NaN
2,Broadcom,Infrastructure Partner 2026 NPA,42000.0,0.0,NaN


fair_value_rollforward.csv: 98 rows
deferred_revenue_rollforward.csv: 10 rows


### 3f. Equity: preferred stack, share reserves, options
- **Preferred stock by series** (144 at 2025-12-31, 145 at 2026-09-30): shares in thousands, issue price, carrying value, liquidation preference.
  Original issue prices trace the valuation path from Series A ($2.57) to Series H ($589.01). These are **pre-split**:
  the 10-for-1 split on Sep 17 2026 means each preferred share converts into 10 common, so Series H is $58.90 per common share
  (`issue_price_per_common_usd`). Common counts, options and EPS are post-split.
- **Common shares reserved** (146, thousands, as-converted) and **anti-dilutive securities** (160): the inputs for a fully diluted share count.
- **Stock option roll-forward** (147).

In [8]:
pref = pd.concat([wide(144).assign(as_of='2025-12-31'), wide(145, 0).assign(as_of='2026-09-30')])
pref.columns = ['series', 'shares_authorized_k', 'shares_outstanding_k', 'issue_price_usd', 'carrying_value_musd', 'liq_pref_musd', 'as_of']
# Sep 17 2026 10-for-1 split (Note 1): preferred counts/prices are pre-split; each preferred share converts into 10 common
pref['issue_price_per_common_usd'] = (pref['issue_price_usd'] / 10).round(4)
pref.to_csv(DATA / 'preferred_stock.csv', index=False); print('preferred_stock.csv:', len(pref), 'rows')

shares = pd.concat([tidy(146, P3_BS, unit='thousands_shares').assign(schedule='common_reserved_as_converted'),
                    tidy(160, P5, unit='thousands_shares').assign(schedule='antidilutive_excluded')])
save(shares[['schedule', 'line_item', 'period', 'unit', 'value']], 'share_counts')

opt = wide(147)
opt.columns = ['event', 'shares_k', 'wavg_exercise_price_usd', 'wavg_life_years', 'intrinsic_value_musd'][:len(opt.columns)]
opt.to_csv(DATA / 'stock_options.csv', index=False); print('stock_options.csv:', len(opt), 'rows')

preferred_stock.csv: 38 rows
share_counts.csv: 124 rows
stock_options.csv: 15 rows


## 4. Narrative KPIs (hand-entered)
These live in prose or images, not tables, so they're typed in here with the line of the filing they came from.
`approx`: `~` = "approximately", `>` = "more than", `<` = "less than". The ≥$100K customer history at year-ends comes
from both the MD&A text and the chart image `mda2ba.jpg`.

In [9]:
K = [
    # metric, as_of / period, value, unit, approx, source quote
    ('customers_100k_ttm', '2023-12-31', 20, 'count', '~', 'MD&A Key factors: grew from approximately 20 as of December 31, 2023'),
    ('customers_100k_ttm', '2024-12-31', 200, 'count', '~', '... to approximately 200 as of December 31, 2024'),
    ('customers_100k_ttm', '2025-09-30', 950, 'count', '>', 'from more than 950 [Sep 30 2025] to more than 11,900'),
    ('customers_100k_ttm', '2025-12-31', 1500, 'count', '~', '... to approximately 1,500 as of December 31, 2025'),
    ('customers_100k_ttm', '2026-09-30', 11900, 'count', '>', '... and to more than 11,900 as of September 30, 2026'),
    ('revenue_from_100k_customers', 'FY2023', 37, 'millions_usd', '~', 'Total revenue from these customers grew from approximately $37 million'),
    ('revenue_from_100k_customers', 'FY2024', 220, 'millions_usd', '~', '... to $220 million'),
    ('revenue_from_100k_customers', 'FY2025', 3500, 'millions_usd', '~', '... to $3.5 billion'),
    ('revenue_from_100k_customers', 'TTM-2026-09-30', 26100, 'millions_usd', '~', '... to $26.1 billion'),
    ('pct_revenue_from_100k_customers', 'FY2023', 90, 'pct', '', 'represented 90%, 57%, 76%, and 74% of total revenue'),
    ('pct_revenue_from_100k_customers', 'FY2024', 57, 'pct', '', 'idem'),
    ('pct_revenue_from_100k_customers', 'FY2025', 76, 'pct', '', 'idem'),
    ('pct_revenue_from_100k_customers', 'TTM-2026-09-30', 74, 'pct', '', 'idem'),
    ('customers_under_100k', 'TTM-2026-09-30', 40_000_000, 'count', '~', 'remaining 26% ... generated by approximately 40 million customers'),
    ('customers_1m_ttm', '2026-09-30', 1900, 'count', '>', 'more than 1,900 customers with $1 million or more of revenue on a TTM basis'),
    ('customers_10m_ttm', '2026-09-30', 250, 'count', '~', 'approximately 250 customers with $10 million or more'),
    ('net_revenue_retention', '2026-09-30', 900, 'pct', '~', 'Net Revenue Retention of approximately 900% as of September 30, 2026'),
    ('pct_100k_customers_platform_and_app', 'Q4-2025', 30, 'pct', '~', 'up from approximately 30% for the three months ended December 31, 2025'),
    ('pct_100k_customers_platform_and_app', 'Q3-2026', 50, 'pct', '>', 'over 50% of our customers with $100,000 or more ... used Claude Platform and at least one first-party app'),
    ('pct_revenue_outside_top5', 'Q1-2026', 64, 'pct', '~', 'up from approximately 64% for the three months ended March 31, 2026'),
    ('pct_revenue_outside_top5', 'Q3-2026', 79, 'pct', '~', 'Approximately 79% of our revenue came from customers outside of our top five customers'),
    ('pct_fortune500_are_100k_customers', '2026-09-30', 50, 'pct', '>', 'Over 50% of the Fortune 500 ... were customers with $100,000 or more'),
    ('pct_global2000_are_100k_customers', '2026-09-30', 30, 'pct', '>', 'over 30% of the Global 2000'),
    ('remaining_performance_obligations', '2026-09-30', 13800, 'millions_usd', '', 'Note 3: remaining performance obligations amounted to $13.8 billion'),
    ('rpo_pct_next_12m', '2026-09-30', 75, 'pct', '~', 'expects to recognize approximately 75% ... over the next 12 months'),
    ('employees', '2026-09-30', 5300, 'count', '~', 'Our Talent: approximately 5,300 employees worldwide'),
    ('enterprise_monthly_spend_non_claude_code_user', 'Q3-2026', 75, 'usd_per_month', '<', 'Claude Enterprise users who did not use Claude Code spent less than $75 per month'),
    ('enterprise_spend_multiple_top5pct_claude_code', 'Q3-2026', 20, 'multiple', '~', 'top 5% of Claude Enterprise users who used Claude Code spent approximately 20 times that amount'),
    # --- revenue quality and cost lines (MD&A) ---
    ('revenue_from_policy_violators_pct_low', '9M-2026', 5, 'pct', '~', 'revenue from such customers represented approximately 5% to 7% of our revenue for the nine months ended September 30, 2026'),
    ('revenue_from_policy_violators_pct_high', '9M-2026', 7, 'pct', '~', 'idem; enforcement enhanced in August and September 2026'),
    ('short_term_compute_expense', '9M-2026', 6500, 'millions_usd', '', '$6.5 billion of compute under short-term commitments ... costs significantly more than multi-year'),
    ('channel_partner_fee_increase_yoy', '9M-2026', 4800, 'millions_usd', '', 'S&M&P increase primarily driven by a $4.8 billion increase in fees paid to channel partners'),
    ('channel_partner_fee_increase_yoy', 'FY2025', 341, 'millions_usd', '', 'a $341 million increase in fees paid to channel partners'),
    ('snm_partnerships_pct_revenue', 'Q1-2026', 22, 'pct', '', 'S&M&P as a percentage of revenue decreased from 22% for Q1-2026 to 14% for Q3-2026 (less channel mix, new partner terms)'),
    ('accumulated_deficit_from_note_marks', '2026-09-30', 135100, 'millions_usd', '', 'accumulated deficit of $156.2 billion, which includes $135.1 billion related to changes in the fair value of our convertible notes'),
    # --- stock comp still to come (Critical accounting estimates) ---
    ('rsu_unrecognized_sbc', '2026-09-30', 32300, 'millions_usd', '~', '$32.3 billion of total unrecognized stock-based compensation expense related to RSUs'),
    ('rsu_sbc_catchup_at_ipo', '2026-09-30', 8200, 'millions_usd', '~', 'Of that amount, $8.2 billion relates to RSUs for which the service-based vesting condition had been satisfied'),
    ('option_unrecognized_sbc', '2026-09-30', 1800, 'millions_usd', '~', '$1.8 billion of unrecognized cost related to unvested stock options ... over 1.2 years'),
    # --- compute commitments beyond the table (Contractual Obligations) ---
    ('commitments_all_in', '2026-09-30', 895400, 'millions_usd', '~', 'table plus short-term and subsequent commitments and leases not yet commenced are approximately $895.4 billion'),
    ('commitments_short_term_min', '2026-09-30', 17400, 'millions_usd', '~', 'short-term commitments ... Minimum payments under such contracts were approximately $17.4 billion'),
    ('leases_not_commenced', '2026-09-30', 451200, 'millions_usd', '~', 'leases executed and not yet commenced ... approximately $451.2 billion; commence 2026-2028, weighted-average term ~10 years'),
    ('commitments_after_sep30', '2026-09-30', 100200, 'millions_usd', '~', 'commitments entered into after September 30, 2026 ... approximately $100.2 billion'),
    ('commitments_after_sep30_short_term', '2026-09-30', 6600, 'millions_usd', '~', 'agreements after September 30, 2026 that are short-term ... approximately $6.6 billion'),
    ('commitments_variable_low', '2026-09-30', 150000, 'millions_usd', '~', 'arrangements without fixed dollar commitment ... we currently estimate such payments to be approximately $150 billion to $175 billion'),
    ('commitments_variable_high', '2026-09-30', 175000, 'millions_usd', '~', 'idem'),
    ('commitments_contingent_or_terminable', '2026-09-30', 213800, 'millions_usd', '~', 'Approximately $213.8 billion ... subject to either exercisable termination rights or contingent conditions'),
    ('commitments_due_next_12m', '2026-09-30', 69800, 'millions_usd', '~', 'Approximately $69.8 billion ... expected to be paid in the 12 months following September 30, 2026'),
    ('commitments_before_2031', '2026-09-30', 447500, 'millions_usd', '~', 'These commitments include approximately $447.5 billion expected to be incurred before December 31, 2030, and substantially all of the remainder ... between 2031 and 2045'),
    ('commit_microsoft', '2026-09-30', 44900, 'millions_usd', '>', 'Microsoft: at least $44.9 billion between November 2026 and June 2033'),
    ('commit_google_csp', '2026-09-30', 121300, 'millions_usd', '>', 'Google CSP hosting: at least approximately $121.3 billion between April 2026 and July 2033'),
    ('commit_google_tpu_variable', '2026-09-30', 137500, 'millions_usd', '~', 'additional TPU compute delivered 2028-2031: approximately $125.0 billion to $150.0 billion (midpoint)'),
    ('commit_amazon', '2026-09-30', 110000, 'millions_usd', '>', 'Amazon: at least $110.0 billion between May 2026 and April 2036'),
    ('commit_spacexai', '2026-09-30', 120500, 'millions_usd', '<', 'SpaceXAI: up to $120.5 billion into 2029, ~$3.1 billion per month at full capacity; cancellable on 90 days notice after early 2027'),
    ('commit_amd_variable', '2026-09-30', 20000, 'millions_usd', '>', 'AMD Helios: expect aggregate expenditures to exceed $20.0 billion'),
    ('commit_nscale', '2026-09-30', 44600, 'millions_usd', '<', 'Nscale: up to approximately $44.6 billion over terms extending into 2033'),
    ('commit_lambda', '2026-09-30', 41300, 'millions_usd', '~', 'Lambda: approximately $41.3 billion over terms extending into 2034'),
    ('commit_lambda_aug2026', '2026-09-30', 34900, 'millions_usd', '~', 'Under our August 2026 agreement, which represents approximately $34.9 billion of the aggregate commitment, either party may terminate if Lambda does not timely secure the facility lease'),
    ('commit_broadcom_leases', '2026-09-30', 161200, 'millions_usd', '~', 'Broadcom TPU equipment leases (finance leases): rent of approximately $161.2 billion excluding interest'),
    ('credit_facility_available', '2026-09-30', 14800, 'millions_usd', '', '$14.8 billion available under the A&R Credit Facility ($15.0B revolver)'),
    # Cap table and future dilution (prose, balance sheet text and MD&A NPA tables)
    ('class_a_shares_outstanding', '2026-09-30', 2168788, 'thousands_shares', '', 'Balance sheet: 2,168,788 [thousand Class A] shares issued and outstanding as of September 30, 2026'),
    ('class_n_shares_outstanding', '2026-09-30', 30643, 'thousands_shares', '', 'Balance sheet: 30,643 [thousand Class N] shares issued and outstanding as of September 30, 2026 (excludes the Public Benefit Fund shares)'),
    ('public_benefit_fund_shares', '2026-06-30', 846841, 'thousands_shares', '', 'contributed 846,840,960 shares of our Class N common stock, representing 5% of our fully diluted shares as of June 30, 2026'),
    ('public_benefit_fund_pct_fd', '2026-06-30', 5, 'pct', '', 'representing 5% of our fully diluted shares as of June 30, 2026 (assuming full conversion of the Amazon Notes)'),
    ('public_benefit_issuance_pct', 'post-IPO', 2, 'pct', '<', 'up to an additional 2% of our then-outstanding shares (plus options, RSUs, convertibles and Fund shares) each time our share price appreciates 50% above the prior share price milestone'),
    ('public_benefit_price_step_pct', 'post-IPO', 50, 'pct', '', 'each time our share price appreciates 50% above the prior share price milestone, beginning with our initial public offering price'),
    ('founder_pledge_pct', 'post-IPO', 80, 'pct', '', 'our co-founders have each pledged to dedicate 80% of their personal equity in Anthropic to philanthropic causes'),
    ('donation_matching_expense', '9M-2026', 2100, 'millions_usd', '', 'expenses related to our donation matching program were $2.1 billion, approximately 6% of total operating expenses'),
    ('equity_plan_evergreen_pct', 'post-IPO', 5, 'pct', '<', '2026 Plan annual increase: the lesser of 5% of the total number of shares of our common stock outstanding'),
    ('espp_evergreen_pct', 'post-IPO', 1, 'pct', '<', 'ESPP annual increase: the lesser of 1% of the total number of shares of our common stock'),
    ('amazon_ownership_cap_pct', '2026-09-30', 12, 'pct', '', 'no portion of a note converts to the extent that Amazon would hold more than 12% of our capitalization (waivable by Amazon)'),
    ('google_ownership_cap_pct', '2026-09-30', 14.99, 'pct', '', "Google's ownership of us [may not] exceed 14.99% of our outstanding capitalization on a post-money, as-converted basis"),
    ('google_ipo_private_placement', 'IPO', 10000, 'millions_usd', '<', 'option to purchase up to $10.0 billion of our Class N common stock in a private placement concurrent with this offering at the IPO price (reduces the Google 2026 NPA)'),
    ('amd_share_purchase_commitment', '2026-09-30', 5000, 'millions_usd', '<', 'AMD has provided a commitment to purchase up to $5.0 billion of our Class N common stock ... at the 30-day VWAP'),
    ('broadcom_npa_conversion_pct_of_vwap', 'post-IPO', 90, 'pct', '', 'convert ... at a conversion price equal to 90% of the 30-day trailing VWAP (85% if revenue performance conditions are not met)'),
    ('broadcom_npa_pik_rate', 'post-IPO', 9, 'pct', '', 'Infrastructure Partner 2026 NPA: 9.00% per annum, paid in kind (13.00% if revenue performance conditions are not met)'),
    ('channel_partner_fee_increase', '9M-2026', 4800, 'millions_usd', '~', 'increase was primarily driven by a $4.8 billion increase in fees paid to channel partners (9M-2025 S,M&P total was $322M, so 9M-2026 fees are $4.8-5.1B)'),
    ('short_term_compute_q3', 'Q3-2026', 5000, 'millions_usd', '', 'delivery of additional compute, including $5.0 billion of compute under short-term commitments, which currently costs significantly more than comparable compute procured under our multi-year commitments'),
    ('price_reduction_revenue_impact', 'Q3-2026', 600, 'millions_usd', '~', 'decrease of approximately $600 million from price reductions introduced as part of new model versions and contractual discounts (Q2 to Q3 2026)'),
    ('enterprise_non_code_user_spend', 'Q3-2026', 75, 'usd_per_month', '<', 'Claude Enterprise users who did not use Claude Code spent less than $75 per month on average'),
    ('enterprise_top5pct_code_user_multiple', 'Q3-2026', 20, 'x', '~', 'top 5% of Claude Enterprise users who used Claude Code spent approximately 20 times that amount'),
    ('revenue_growth_yoy_q3', 'Q3-2026', 1127, 'pct', '', 'our revenue grew 1,127% year over year for the three months ended September 30, 2026'),
    ('compute_growth_yoy_q3', 'Q3-2026', 382, 'pct', '', 'compared to our compute and infrastructure expenses that grew 382% during the same period'),
    ('knowledge_work_value', 'FY2025', 40000000, 'millions_usd', '>', 'the work that knowledge workers perform represented more than $40 trillion in economic value globally in 2025'),
    ('it_spend', 'FY2025', 4300000, 'millions_usd', '~', 'information technology spend grew from approximately $1.2 trillion in 2006 to $4.3 trillion in 2025'),
    ('it_spend_cagr_2006_2025', 'FY2025', 7, 'pct', '~', 'a CAGR of approximately 7%'),
    ('knowledge_workers_2028', '2028', 1000, 'millions_people', '>', 'knowledge workers globally, a population that is estimated to exceed one billion in 2028'),
    ('vested_matching_obligation_shares', '2026-09-30', 553.441297, 'millions_shares', '<', 'of which 553,441,297 shares represent the maximum number of shares issuable as matching contributions on employee donations that may be made based on employee pledges on vested equity awards outstanding as of September 30, 2026, less the Matching Issuance (the "Vested Matching Obligation")'),
    ('cloud_spend', 'FY2025', 1900000, 'millions_usd', '', 'cloud computing spend grew from zero to $1.9 trillion over the same period'),
]
kpis = pd.DataFrame(K, columns=['metric', 'period', 'value', 'unit', 'approx', 'source'])
kpis.to_csv(DATA / 'kpis.csv', index=False)
print('kpis.csv:', len(kpis), 'rows')

kpis.csv: 88 rows


## 5. Reconciliation checks
If a table index shifted or a parse went wrong, one of these fails. Tolerance is $2M for rounding.

In [10]:
def v(df, item, period, **kw):
    m = (df.line_item == item) & (df.period == period)
    for k, x in kw.items(): m &= df[k] == x
    assert m.sum() == 1, (item, period, m.sum())
    return df.loc[m, 'value'].item()
close = lambda a, b, tol=2: abs(a - b) <= tol

opex = ['Compute and infrastructure', 'Personnel', 'Bartz settlement', 'Sales, marketing, and partnerships', 'Other operating and administrative']
for p in P5:
    assert close(sum(v(inc, x, p) for x in opex), v(inc, 'Total operating expenses', p)), p
    assert close(v(inc, 'Revenue', p) - v(inc, 'Total operating expenses', p), v(inc, 'Operating loss', p)), p
    assert close(v(inc, 'Loss before provision for income taxes', p) - v(inc, 'Provision for income taxes', p), v(inc, 'Net loss', p)), p
    for d in ['geography', 'pricing_model']:
        assert close(rev[(rev.dimension == d) & (rev.period == p) & (rev.line_item != 'Total')].value.sum(), v(inc, 'Revenue', p)), (d, p)
    ops = ['Net cash provided by (used in) operating activities', 'Net cash used in investing activities', 'Net cash provided by financing activities',
           'Effect of exchange rate changes on cash, cash equivalents, and restricted cash']
    assert close(sum(v(cf, x, p) for x in ops), v(cf, 'Net increase in cash, cash equivalents, and restricted cash', p)), p

for p in P3_BS:
    assert close(v(bs, 'Total liabilities', p) + v(bs, bs.line_item[bs.line_item.str.startswith('Redeemable')].iloc[0], p)
                 + v(bs, 'Total stockholders’ deficit', p), v(bs, 'Total assets', p)), p

# quarters add up to the annual / nine-month figures
for item in ['Revenue', 'Total operating expenses', 'Net loss']:
    assert close(sum(v(q, item, f'Q{i}-2025') for i in range(1, 5)), v(inc, item, 'FY2025'), 3), item
    assert close(sum(v(q, item, f'Q{i}-2026') for i in range(1, 4)), v(inc, item, '9M-2026'), 3), item

assert co.query('due != "total"').groupby('line_item').value.sum().sub(co.query('due == "total"').set_index('line_item').value).abs().max() <= 2
print('All reconciliation checks passed.')

All reconciliation checks passed.


## Findings
1. Revenue: $41M (FY23) → $386M → $4.6B (FY25) → $33.2B in the first nine months of 2026; Q3-2026 alone was $17.0B (§3a).
2. GAAP net loss of $104B in 9M-2026 is almost entirely the $94B fair-value mark on convertible notes plus $10B on other
   financial instruments; Adjusted Operating Income turned positive ($4.1B, 12% margin) (§3b).
3. $314B of compute purchase commitments against $132B of cash and short-term investments (§3e).
4. Preferred issue prices run from $0.26 to $58.90 per common share (Series A → Series H, split-adjusted) (§3f).

**Caveats:** offering terms, pro forma capitalization, dilution, ownership percentages and Run-Rate Revenue are blank in this
amendment. Customer concentration `*` cells mean "below 10%", not zero.

**Open items:** re-run on the priced amendment; add the beneficial-ownership and lock-up tables once their numbers are filled in.